In [37]:
# IMPORTS
import pandas as pd
import hashlib
import os
from datetime import datetime

In [38]:
# FUNÇÕES ÚTEIS
def normalize_strings(df: pd.DataFrame) -> pd.DataFrame:
    # Remove espaços extras, converte strings para maiúsculas e preenche NaN com ''
    df = df.copy()
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip().str.upper().fillna('')
    return df

def normalize_dates(df: pd.DataFrame, date_cols: list) -> pd.DataFrame:
    # Converte colunas de datas para datetime, mantém NaT se inválido
    df = df.copy()
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce', format='%Y%m%d')
    return df

In [39]:
# FUNÇÕES
def row_hash_sha256(row: pd.Series) -> str:
    row_str = '|'.join(row.astype(str).fillna(''))
    return hashlib.sha256(row_str.encode('utf-8')).hexdigest()

def get_latest_bronze(dataset: str) -> pd.DataFrame:
    folder = os.path.join("..", "data", "01_bronze")
    files = sorted([f for f in os.listdir(folder) if f.startswith(dataset)])
    if not files:
        raise FileNotFoundError(f"Nenhum arquivo encontrado para {dataset} em {folder}")
    latest_file = files[-1]
    df = pd.read_parquet(os.path.join(folder, latest_file))
    return df

def save_silver(df: pd.DataFrame, dataset: str) -> str:
    folder = os.path.join("..", "data", "02_silver")
    os.makedirs(folder, exist_ok=True)
    timestamp = datetime.now().strftime("%d%m%Y")
    path = os.path.join(folder, f"{dataset}_{timestamp}.parquet")
    df.to_parquet(path, index=False, engine="pyarrow")
    return path

def scd2_update(dim: pd.DataFrame, df_new: pd.DataFrame, key_cols: list) -> pd.DataFrame:
    # Atualiza dimensão com SCD2 baseado em hash.
    df_new = df_new.copy()
    df_new['current_flag'] = True
    df_new['start_date'] = datetime.now()
    df_new['end_date'] = pd.NaT

    if dim is None or dim.empty:
        return df_new

    # Marcar registros que mudaram
    merged = df_new.merge(dim, on=key_cols, how='left', suffixes=('', '_old'))
    mask_changed = merged['hash_old'].notna() & (merged['hash'] != merged['hash_old'])

    # Atualizar current_flag e end_date na dimensão antiga
    for idx, row in merged[mask_changed].iterrows():
        dim_idx = dim[(dim[key_cols] == row[key_cols]).all(axis=1) & (dim['current_flag']==True)].index
        dim.loc[dim_idx, 'current_flag'] = False
        dim.loc[dim_idx, 'end_date'] = datetime.now()

    # Adicionar novas versões
    dim = pd.concat([dim, df_new], ignore_index=True)
    return dim

In [40]:
# CARREGAR BRONZE
df_medicamentos = get_latest_bronze("Medicamentos")
df_notificacoes = get_latest_bronze("Notificacoes")
df_reacoes = get_latest_bronze("Reacoes")

In [41]:
# NORMALIZAÇÃO
df_medicamentos = normalize_strings(df_medicamentos)
df_notificacoes = normalize_strings(df_notificacoes)
df_reacoes = normalize_strings(df_reacoes)

date_cols_notificacoes = ['DATA_INCLUSAO_SISTEMA','DATA_ULTIMA_ATUALIZACAO','DATA_NOTIFICACAO', 'DATA_NASCIMENTO', 'DATA_INICIO_HORA','DATA_FINAL_HORA']
df_notificacoes = normalize_dates(df_notificacoes, date_cols_notificacoes)
df_reacoes = normalize_dates(df_reacoes, ['DATA_INICIO_HORA','DATA_FINAL_HORA'])

In [42]:
# CRIAR HASH 
df_medicamentos['hash'] = df_medicamentos.apply(row_hash_sha256, axis=1)
df_notificacoes['hash'] = df_notificacoes.apply(row_hash_sha256, axis=1)
df_reacoes['hash'] = df_reacoes.apply(row_hash_sha256, axis=1)

In [43]:
# DIMENSÕES
# dim_medicamentos
dim_med_path = os.path.join("data", "02_silver", "dim_medicamentos.parquet")
dim_medicamentos = pd.read_parquet(dim_med_path) if os.path.exists(dim_med_path) else pd.DataFrame()
dim_medicamentos = scd2_update(dim_medicamentos, df_medicamentos, key_cols=['IDENTIFICACAO_NOTIFICACAO'])
save_silver(dim_medicamentos, "dim_medicamentos")
print(f"dim_medicamentos atualizado ({len(df_medicamentos)} linhas processadas)")

# dim_reacoes
dim_reac_path = os.path.join("data", "02_silver", "dim_reacoes.parquet")
dim_reacoes = pd.read_parquet(dim_reac_path) if os.path.exists(dim_reac_path) else pd.DataFrame()
dim_reacoes = scd2_update(dim_reacoes, df_reacoes, key_cols=['IDENTIFICACAO_NOTIFICACAO'])
save_silver(dim_reacoes, "dim_reacoes")
print(f"dim_reacoes atualizado ({len(df_reacoes)} linhas processadas)")

# FATOS
# fat_notificacoes
fat_notificacoes = df_notificacoes.copy()
fat_notificacoes = fat_notificacoes.merge(
    dim_medicamentos[['IDENTIFICACAO_NOTIFICACAO','hash']],
    on='IDENTIFICACAO_NOTIFICACAO',
    how='left',
    suffixes=('','_med')
)
fat_notificacoes = fat_notificacoes.merge(
    dim_reacoes[['IDENTIFICACAO_NOTIFICACAO','hash']],
    on='IDENTIFICACAO_NOTIFICACAO',
    how='left',
    suffixes=('','_reac')
)
save_silver(fat_notificacoes, "fat_notificacoes")
print(f"fat_notificacoes atualizado ({len(fat_notificacoes)} linhas processadas)")

dim_medicamentos atualizado (552887 linhas processadas)
dim_reacoes atualizado (815877 linhas processadas)
fat_notificacoes atualizado (2063878 linhas processadas)
